In [6]:
!nvidia-smi
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

Thu Nov  5 09:02:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

AttributeError: module 'torch' has no attribute 'cuda'

In [3]:
!pip install -q -U transformers peft bitsandbytes trl accelerate datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [4]:
#huggingface login
from google.colab import userdata
print(userdata.get("HF_TOKEN")[:3])   # should print: hf_

hf_


In [5]:
#settings
model_name = "NousResearch/Llama-2-7b-chat-hf"   # open mirror, no access approval needed
dataset_name = "mlabonne/guanaco-llama2-1k"       # 1,000 samples, already in Llama-2 format
new_model = "Llama-2-7b-chat-finetune"

# LoRA
lora_r = 64
lora_alpha = 16
lora_dropout = 0.1

In [6]:
#Load the dataset
from datasets import load_dataset

dataset = load_dataset(dataset_name, split="train")
print(dataset)
print(dataset[0]["text"][:500])

README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…): reconstructing file:   0%|          |  0.00B /  967kB            

data/train-00000-of-00001-9ad84bb9cf65a4(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 1000
})
<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal


In [7]:
import gc, torch
try:
    del model, trainer
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=torch.float16,          # the fix: force float16
)
model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

In [8]:
#LoRA Config
from peft import LoraConfig

peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

In [9]:
#training settings
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    weight_decay=0.001,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,
    logging_steps=25,
    fp16=True,
    bf16=False,
    max_length=512,
    dataset_text_field="text",
    report_to="tensorboard",
)

In [10]:
#Train
import torch
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

# Force every trainable weight to float32 (the fix)
for name, p in trainer.model.named_parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)

# Check: this should print only float32
print({p.dtype for p in trainer.model.parameters() if p.requires_grad})

trainer.train()

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

{torch.float32}


Step,Training Loss
25,1.553766
50,1.382028
75,1.273734
100,1.271784
125,1.226795


TrainOutput(global_step=125, training_loss=1.3416213226318359, metrics={'train_runtime': 1276.2964, 'train_samples_per_second': 0.784, 'train_steps_per_second': 0.098, 'total_flos': 1.964099140632576e+16, 'train_loss': 1.3416213226318359, 'epoch': 1.0})

In [11]:
#save the adapters
trainer.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)

!ls -lh {new_model}

total 260M
-rw-r--r-- 1 root root 1.2K Nov  5 08:52 adapter_config.json
-rw------- 1 root root 257M Nov  5 08:52 adapter_model.safetensors
-rw-r--r-- 1 root root 5.2K Nov  5 08:51 README.md
-rw-r--r-- 1 root root  398 Nov  5 08:52 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Nov  5 08:52 tokenizer.json


In [12]:
#watch the loss curve
%load_ext tensorboard
%tensorboard --logdir results/runs

Output hidden; open in https://colab.research.google.com to view.

In [13]:
#Test the fine-tuned model
from transformers import pipeline, logging
logging.set_verbosity(logging.CRITICAL)

prompt = "What is a large language model?"
pipe = pipeline(task="text-generation", model=trainer.model, tokenizer=tokenizer, max_new_tokens=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]["generated_text"])

<s>[INST] What is a large language model? [/INST]A[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[[


In [14]:
#Free GPU memory
import gc
del model, pipe, trainer
gc.collect(); gc.collect()
torch.cuda.empty_cache()

In [1]:
import torch
print(torch.__version__, torch.cuda.is_available())

2.11.0+cu130 True


In [2]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

memory.used [MiB], memory.total [MiB]
3 MiB, 15360 MiB


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_name = "NousResearch/Llama-2-7b-chat-hf"
new_model = "Llama-2-7b-chat-finetune"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb, device_map={"": 0}, torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(base, new_model)
tok = AutoTokenizer.from_pretrained(new_model)

inputs = tok("<s>[INST] What is a large language model? [/INST]",
             return_tensors="pt", add_special_tokens=False).to("cuda")
out = model.generate(**inputs, max_new_tokens=200)
print(tok.decode(out[0], skip_special_tokens=True))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[INST] What is a large language model? [/INST] A large language model is a type of artificial intelligence model that is trained on a large dataset of text, typically billions of words, to generate language outputs that are coherent and realistic. This type of model is designed to learn patterns and relationships in language, and can be used for a wide range of applications, such as language translation, text summarization, and chatbots.

Large language models are typically trained using a variant of the transformer architecture, which is a type of neural network that is particularly well-suited for natural language processing tasks. The transformer architecture uses self-attention mechanisms to allow the model to "attend" to different parts of the input text, and generate outputs that are contextually appropriate.

Some examples of large language models include BERT, RoBERTa, and XLNet. These models have been trained on large datasets of text, and have been shown to achieve state-of-t

In [5]:
from google.colab import drive
   drive.mount('/content/drive')
   !cp -r Llama-2-7b-chat-finetune /content/drive/MyDrive/

IndentationError: unexpected indent (2492369527.py, line 2)

In [6]:
#Merge the adapters into a standalone model
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
model = PeftModel.from_pretrained(base_model, new_model)
model = model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

OutOfMemoryError: CUDA out of memory. Tried to allocate 12.16 GiB. GPU 0 has a total capacity of 14.56 GiB of which 10.15 GiB is free. Including non-PyTorch memory, this process has 4.41 GiB memory in use. Of the allocated memory 4.16 GiB is allocated by PyTorch, and 130.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
#optional = Push to HuggingFace
repo = "Ashiq001/Llama-2-7b-chat-finetune"

model.push_to_hub(repo)
tokenizer.push_to_hub(repo)